# Optimization of the Intermediate Representation

In this notebook, we will walk you through optimizing an intermediate representation (IR) using `qret opt`.
There are two ways to run `qret opt`:

- Run the optimizations directly by chaining `--pass` arguments.
- Execute pass and external commands from a YAML pipeline.

The goals of this chapter are:

- Compare `Call` inlining (single-level and recursive).
- Decompose high-level instructions into low-level instructions using `ir::decompose_inst`.
- Statically simplify branches using `ir::static_condition_pruning`.
- Connect an external pass to the pipeline.


First, we prepare the execution environment and input files.
Change `exe_path` and `GRIDSYNTH_PATH` according to your environment. Make sure that gridsynth has necessary execution permission as well.

In [ ]:
import pathlib
import platform

import graphviz
from IPython.display import Code
import os

project_root = pathlib.Path("../../../..").resolve()
qret_path = project_root / "build" / "main"
if platform.system() == "Darwin": 
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth_macos"
else:
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth"

data_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_3.json"

os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

First make sure that `qret` and `gridsynth` are executable and can read the input JSON.

In [ ]:
!qret --version

In [ ]:
!{gridsynth_path} --help

In [ ]:
Code(filename=data_path, language="JSON")

Next, list the options of the `opt` command.

There are two ways to run `qret opt`:

- **Direct argument specification**: Specify `--pass` multiple times and apply them sequentially in one execution.
- **Pipeline file**: Defines the order and input/output in YAML and executes it.

We can check the available options with `--help`.


In [ ]:
!qret opt --help

## Check Optimization Target

First, use `print -s` to check the list of functions in the target module, and select `Tutorial3Function` as the optimization target.

In [ ]:
!qret print -i {data_path} -s

In [ ]:
!qret diagram -i {data_path} --function "Tutorial3Function" --graph-format "CallGraph" --display_num_calls -o { output_dir / "tutorial_3_diagram_call_graph.dot"}

graphviz.Source.from_file(output_dir / "tutorial_3_diagram_call_graph.dot")

Next, we print the unoptimized `Tutorial3Function` and check the sequence of `Call` instructions.

In [ ]:
!qret print -i {data_path} -f "Tutorial3Function"

Next, use `-d 2` to expand to the call destination. By comparing with the previous output we see the difference before and after expansion.

In [ ]:
!qret print -i {data_path} -f "Tutorial3Function" -d 2

## Applying the Optimization Passes In Quration

Here, we apply the built-in passes in order and compare the difference using the `print` output.
In this tutorial we will look at the following passes: `ir::inliner` -> `ir::recursive_inliner` -> `ir::decompose_inst` -> `ir::static_condition_pruning`.


For copmleteness, the main built-in passes that can be called directly with `--pass` are:

- `ir::inliner`: Expands `Call` instructions by one level
- `ir::recursive_inliner`: Expands `Call` instructions recursively
- `ir::decompose_inst`: Decomposes high-level instructions into low-level instructions
- `ir::static_condition_pruning`: Simplifies statically determinable branches
- `ir::ignore_global_phase`: Removes `GlobalPhase`
- `ir::delete_consecutive_same_pauli`: Removes consecutive applications of the same Pauli.
- `ir::delete_opt_hint`: Removes optimization hint instruction
- `ir::external`: Connects to an external optimization pass


### Inline Expansion (One-Stage Expansion of `Call` Instructions)

`ir::inliner` inlines `Call` isntructions in the specified function by one level.
Let us see what the function looks like after this pass.


In [ ]:
!qret opt -i {data_path} -o { output_dir / "tutorial_3_inlined.json"} -f "Tutorial3Function" --pass "ir::inliner"
!qret print -i  { output_dir / "tutorial_3_inlined.json"} -f "Tutorial3Function" -d 2

### Recursive Inlining (Recursive Substitution of `Call` Instructions)

`ir::recursive_inliner` continues to expand `Call` instructions that appears after a previous expansion.
The difference with `ir::inliner` is that it can collapse the entire call chain.


In [ ]:
!qret opt -i {data_path} -o { output_dir / "tutorial_3_recursively_inlined.json"} -f "Tutorial3Function" --pass "ir::recursive_inliner"
!qret print -i { output_dir / "tutorial_3_recursively_inlined.json"} -f "Tutorial3Function" -d 2

### Decomposition of Rotation-Related Commands

`ir::decompose_inst` decomposes high-level instructions into a sequence of lower-level instructions.
This pass requires setting `GRIDSYNTH_PATH`.


In [ ]:
!qret opt -i {data_path} -o { output_dir / "tutorial_3_decomposed.json"} -f "Tutorial3Function" --pass "ir::decompose_inst"
!qret print -i { output_dir / "tutorial_3_decomposed.json"} -f "Tutorial3Function" -d 3

## Static reduction of Conditional Branches
`ir::static_condition_pruning` resolves `Branch` / `Switch` instructions whose outcomes can be statically determined, and prunes any unreachable paths. In this section, we will examine this behavior using the `DiscreteDistribution`, and `Switch` structures from `tutorial_2.json`.


In the `Tutorial2Function` from `tutorial_2.json`, the `registers` for the `Switch` instruction are evaluated in the order `[@r0, @r1]`. Since `r0` represents the LSB, the resulting value becomes `value = 1*r0 + 2*r1`.
Using this mapping, you can easily determine which `case` or `default` branch will be targeted.

In [ ]:
tutorial_2_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_2.json"
!qret print -i {tutorial_2_path} -f "Tutorial2Function"

In [ ]:
!qret opt -i {tutorial_2_path} -o { output_dir / "tutorial_2_pruned.json"} -f "Tutorial2Function" --pass "ir::static_condition_pruning" --ir-static-condition-pruning-seed 0
!qret print -i {output_dir / "tutorial_2_pruned.json"} -f "Tutorial2Function" -d 2

By specifying the `ir-static-condition-pruning-seed` argument, it allows you to reproduce the sampling results of the `DiscreteDistribution` during the optimization pass.
Using the same seed will select the same branch, making it easier to compare experiments.


Run the simulation again with a different seed to verify that the remaining `case` switches accordingly.
To ensure reproducible comparisons, fix the seed to a specific value and record it.

In [ ]:
!qret opt -i {tutorial_2_path} -o { output_dir / "tutorial_2_pruned_seed3.json"} -f "Tutorial2Function" --pass "ir::static_condition_pruning" --ir-static-condition-pruning-seed 3
!qret print -i { output_dir / "tutorial_2_pruned_seed3.json"} -f "Tutorial2Function" -d 2
!qret opt -i {tutorial_2_path} -o { output_dir / "tutorial_2_pruned_seed2.json"} -f "Tutorial2Function" --pass "ir::static_condition_pruning" --ir-static-condition-pruning-seed 2
!qret print -i { output_dir / "tutorial_2_pruned_seed2.json"} -f "Tutorial2Function" -d 2

So far, we have applied four passes and compared the differences using `print`.

- `inliner` / `recursive_inliner`: Control the granularity of `Call` expansion
- `decompose_inst`: Decompose higher level to lower level instructions
- `static_condition_pruning`: Prune branches and unreachable routes


## Chaining External Passes

When using an external implementation, you can chain the `qret` pass and external commands together in a YAML pipeline.
In this example, we run a Python script after `ir::inliner` to optimize the `Tutorial3Function` function.


In [ ]:
tutorial_3_decompose_using_external_pass_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_3_decompose_using_external_pass.yaml"
Code(filename=tutorial_3_decompose_using_external_pass_path, language="YAML")

`data/decompose.py` is a sample script that decomposes `CCX` gates into `H`, `CX`, `T`, and `TDag` gates. When creating your own scripts, please ensure that the input and output JSON formats follow the official Quration structure.

In [ ]:
tutorial_3_decompose_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_3_decompose.py"
Code(filename=tutorial_3_decompose_path, language="Python")

Finally, run `qret opt --pipeline` and inspect the updated `decomposed.json` to verify the changes.

In [ ]:
!qret opt  --pipeline {tutorial_3_decompose_using_external_pass_path}
!qret print -i {output_dir / "tutorial_3_decomposed.json"} -f "Tutorial3Function" -d 3

In this chapter, we reviewed the basic workflow for applying optimization passes using `qret opt`.

The core steps are:
- Establish a baseline using the `print` command first.
- Determine the execution order of the passes based on your optimization goals.
- Integrate external passes into the pipeline as needed.
- Verify the differences by running `print` again after execution.